In [1]:
# Célula — leitura da Silver/Gold produzida pelo dbt (somente leitura)
import duckdb

con = duckdb.connect()                                   # sessão em memória
con.execute("LOAD httpfs;")                              # extensão já instalada na imagem
con.execute("""
    CREATE OR REPLACE SECRET minio (
        TYPE s3, KEY_ID 'minioadmin', SECRET 'minioadmin123',
        ENDPOINT 'host.docker.internal:9000',
        URL_STYLE 'path', USE_SSL false, REGION 'us-east-1'
    );
""")

# Anexa o banco do dbt SEM permissão de escrita.
# O alias precisa ser "lakehouse": as views criadas pelo dbt usam esse nome.
con.execute("ATTACH '../data/lakehouse.duckdb' AS lakehouse (READ_ONLY)")
con.execute("USE lakehouse")

con.sql("""
select cliente_id, segmento_cliente, valido_de, valido_ate, is_atual
from marts.dim_clientes_historico
where cliente_id in (select cliente_id from marts.dim_clientes_historico group by 1 having count(*) > 1)
order by cliente_id, valido_de
limit 10;
""").df()

,cliente_id,segmento_cliente,valido_de,valido_ate,is_atual
0,155,alto_valor,2026-11-12 21:43:21.026703,2026-11-12 21:46:34.353243,False
1,155,vip,2026-11-12 21:46:34.353243,9999-12-31 00:00:00.000000,True
2,193,alto_valor,2026-11-12 21:43:21.026703,2026-11-12 21:46:34.353243,False
3,193,vip,2026-11-12 21:46:34.353243,9999-12-31 00:00:00.000000,True
4,240,alto_valor,2026-11-12 21:43:21.026703,2026-11-12 21:46:34.353243,False
5,240,vip,2026-11-12 21:46:34.353243,9999-12-31 00:00:00.000000,True
6,297,alto_valor,2026-11-12 21:43:21.026703,2026-11-12 21:46:34.353243,False
7,297,vip,2026-11-12 21:46:34.353243,9999-12-31 00:00:00.000000,True
8,309,alto_valor,2026-11-12 21:43:21.026703,2026-11-12 21:46:34.353243,False
9,309,vip,2026-11-12 21:46:34.353243,9999-12-31 00:00:00.000000,True


In [2]:
con.close()